In [1]:
import pandas as pd
df = pd.read_excel("Online Retail.xlsx")

In [2]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Data Cleaning

In [3]:
# checking null values
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [4]:
df = df.dropna(subset=['CustomerID'])

In [5]:
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [6]:
# Filtering: Keep only positive data
df = df[df['Quantity']>0]

### Verification of Cleaned Data

In [13]:
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [9]:
# check if any negative or zero data present 
df['Quantity'].min() 

np.int64(1)

In [12]:
df['UnitPrice'].min()

np.float64(0.0)

In [10]:
df[df['Quantity'] <= 0] 

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


In [14]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

### RFM Feature Engineering

In [15]:
# snapshot date is analysis date, Recency = Analysis Day - Last Purchase Day
import pandas as pd
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [17]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

In [18]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [20]:
# RFM modelling
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})

In [21]:
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalPrice': 'Monetary'
}, inplace=True)

In [22]:
rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


### RFM Scoring

In [26]:
# Divides data into equal sized groups(here 4)
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])

rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])

rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

In [28]:
# Combine and convert to string
rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

### Customer Segmentation Logic

In [29]:
def segment_customer(row):
    if row['RFM_score'] == '444':
        return 'VIP'
    elif row['F_score'] >= 3:
        return 'Loyal'
    elif row['R_score'] <= 2:
        return 'At Risk'
    else:
        return 'Regular'

In [32]:
# apply function on new column
rfm['Segment'] = rfm.apply(segment_customer, axis=1)

In [33]:
rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score,Segment
CustomerID,,,,,,,,
12346.0,326,1,77183.60,1,1,4,114,At Risk
12347.0,2,7,4310.00,4,4,4,444,VIP
12348.0,75,4,1797.24,2,3,4,234,Loyal
12349.0,19,1,1757.55,3,1,4,314,Regular
12350.0,310,1,334.40,1,1,2,112,At Risk


In [34]:
# count how many customers are in each segment
rfm['Segment'].value_counts()

Segment
Loyal      1680
At Risk    1504
Regular     666
VIP         489
Name: count, dtype: int64

### Exporting results

In [36]:
# bring CustomerID back as column
rfm = rfm.reset_index()               

In [38]:
rfm.to_csv("rfm_table.csv", index=False)